In [2]:
import torch
import numpy as np
import equinox as eqx
import jax
import jax.numpy as jnp
from jaxtyping import Array, Float, Int, PyTree  # https://github.com/google/jaxtyping
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
import pandas as pd
import pickle
import os
from itertools import combinations
from tqdm import tqdm

# example of calculating the frechet inception distance
import numpy
from numpy import cov
from numpy import trace
from numpy import iscomplexobj
from numpy.random import random
from scipy.linalg import sqrtm
from scipy.stats import wasserstein_distance_nd


# import ot  # POT library

# def sinkhorn_wasserstein(x, y, reg=0.1):
#     n, m = x.shape[0], y.shape[0]
#     a, b = np.ones(n)/n, np.ones(m)/m
#     M = ot.dist(x, y)  # cost matrix
#     return ot.sinkhorn2(a, b, M, reg)

from main_project.train import train_classifier
from main_project.model import targetClassifier
from main_project.visualize import plot_mmd_image_heatmaps_full, plot_latent_dim_vs_average_mmd, plot_gamma_vs_mmd

from main_project.utils import load
from main_project.environment import MODELS_DIM, INTERMEDIATE_FRACTIONS, MAX_POINTS, GAMMA, LABELS


In [26]:
figure_3 = pd.read_csv("../../data/figure_3.csv").iloc[:, 0:]

In [27]:
figure_3

,latent_dim,0.0001,0.001,0.01,0.1,1.0
0,2,"MMD: 0.3363, W-Dist: 4.0658, Conf: 0.5327","MMD: 0.0838, W-Dist: 2.6385, Conf: 0.8253","MMD: 0.1192, W-Dist: 2.8560, Conf: 0.8085","MMD: 0.2508, W-Dist: 3.4012, Conf: 0.7628","MMD: 0.7149, W-Dist: 4.5156, Conf: 0.5947"
1,8,"MMD: 0.0175, W-Dist: 4.3333, Conf: 0.9036","MMD: 0.0011, W-Dist: 4.1095, Conf: 0.9525","MMD: 0.0159, W-Dist: 4.4143, Conf: 0.9692","MMD: 0.2693, W-Dist: 5.3497, Conf: 0.9944","MMD: 1.1058, W-Dist: 6.9842, Conf: 1.0000"
2,16,"MMD: 0.0013, W-Dist: 4.3809, Conf: 0.9539","MMD: 0.0018, W-Dist: 4.4694, Conf: 0.9592","MMD: 0.0333, W-Dist: 4.8799, Conf: 0.9781","MMD: 0.5830, W-Dist: 6.3282, Conf: 0.9991","MMD: 1.1657, W-Dist: 7.4006, Conf: 1.0000"
3,32,"MMD: 0.0084, W-Dist: 4.4021, Conf: 0.9406","MMD: 0.0009, W-Dist: 4.3015, Conf: 0.9626","MMD: 0.0115, W-Dist: 4.5383, Conf: 0.9781","MMD: 0.2802, W-Dist: 5.3839, Conf: 0.9985","MMD: 1.0433, W-Dist: 6.9550, Conf: 1.0000"


In [ ]:
summary_df = pd.read_csv("../../data/evaluation_summary.csv")



In [5]:
summary_df.shape

(900, 9)

In [4]:


summary_df.head(20)

,latent_dim,gamma,source_label,target_label,mmd_latent,wasserstein_distance_latent,mmd_image,classifier_confidence_image,wasserstein_distance,entropy,running_time,iter_count
0,2,0.1,0,1,0.002853,0.428048,0.750317,0.997960,3.666390,13.656024,0.625846,65
1,2,0.1,0,2,0.028065,1.006158,0.069080,0.748562,3.078469,12.498967,0.195861,77
2,2,0.1,0,3,0.011004,0.623693,0.473702,0.784107,3.774287,13.435592,2.194878,999
3,2,0.1,0,4,0.000316,0.344700,0.226643,0.857422,3.573401,13.189256,0.332329,152
4,2,0.1,0,5,0.006596,0.444535,0.690590,0.031987,5.298658,13.232627,1.208491,383
5,2,0.1,0,6,0.001203,0.298904,0.040941,0.919219,2.133586,12.267374,1.000814,225
6,2,0.1,0,7,0.003547,0.470934,0.331319,0.962205,3.932227,13.155322,0.527096,230
7,2,0.1,0,8,0.000462,0.252997,0.366722,0.531237,3.607937,13.432164,1.212539,348
8,2,0.1,0,9,0.000311,0.202738,0.069145,0.965098,2.117815,13.166410,2.224243,999
9,2,0.1,1,2,0.030776,0.973369,0.122815,0.783570,3.263353,13.011801,0.575304,62
